# Advanced task - contributes to the coursework
## Generate the dataset.json - part 1
Run the code below to generate logs

In [ ]:
import os
import random
import threading
import time
from collections import Counter
from datetime import datetime
from difflib import unified_diff
from typing import Dict, List, Optional, Tuple

# -----------------------------
# Configuration
# -----------------------------
N_THREADS = 10
N_FILES = 10
ITERATIONS = 2000
SLEEP_MIN = 0.001
SLEEP_MAX = 0.01

# Monitor config
MONITOR_POLL_INTERVAL_S = 0.005   # how often to scan the files
MONITOR_TIME_WINDOW_NS = 50_000_000  # 50 ms window to match a write hint
MONITOR_DIFF_MAX_LINES = 50       # limit diff lines written to monitor.txt

FILES = [f"file{i}.txt" for i in range(1, N_FILES + 1)]
FILE_LOCKS = {name: threading.Lock() for name in FILES}

# Logs
LOG_PATH = "logs.txt"          # main event log (from the prior version); change to .txt if desired
MONITOR_LOG_PATH = "monitor.txt"  # new: file-system monitor log

LOG_LOCK = threading.Lock()
MONITOR_LOG_LOCK = threading.Lock()

# Hints set by agents around writes, so the monitor can "guess" the writer thread
LAST_WRITE_HINTS: Dict[str, Dict[str, object]] = {}
LAST_WRITE_LOCK = threading.Lock()


# -----------------------------
# Utilities
# -----------------------------
def ensure_files_exist() -> None:
    for name in FILES:
        if not os.path.exists(name):
            with open(name, "w", encoding="utf-8") as f:
                f.write("")  # start empty


def iso_now() -> str:
    """Return current wall-clock time in ISO 8601 with milliseconds."""
    return datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"


def iso_from_ns(ns: int) -> str:
    """Convert a time_ns() value to ISO 8601 string (approx, using current wall clock for date anchor)."""
    # We can't reverse to absolute easily without a reference; for clarity we'll just return iso_now().
    # If you want exact mtime in ISO, you can compute from os.stat().st_mtime directly.
    return iso_now()


def init_logs() -> None:
    """Create/truncate logs at the start of the run."""
    with LOG_LOCK:
        with open(LOG_PATH, "w", encoding="utf-8") as f:
            f.write(f"# Log initialised at {iso_now()}\n")
    with MONITOR_LOG_LOCK:
        with open(MONITOR_LOG_PATH, "w", encoding="utf-8") as f:
            f.write(f"# Monitor initialised at {iso_now()}\n")


def log_event(message: str) -> None:
    """Thread-safe append to the main event log."""
    with LOG_LOCK:
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(message + "\n")


def monitor_log(message: str) -> None:
    """Thread-safe append to the monitor log."""
    with MONITOR_LOG_LOCK:
        with open(MONITOR_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(message + "\n")


def remove_agent_trace(lines: List[str], agent_id: int) -> List[str]:
    """
    Removes any lines that are exactly this agent's tag.
    Tag format: "agent:<id>\n"
    """
    tag = f"agent:{agent_id}\n"
    return [ln for ln in lines if ln != tag]


def append_blame(lines: List[str], blamed_id: int) -> List[str]:
    """Appends a line that blames another agent."""
    lines.append(f"agent:{blamed_id}\n")
    return lines


def file_read_lines(path: str) -> List[str]:
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.readlines()
    except FileNotFoundError:
        return []


def file_stat_mtime_ns(path: str) -> int:
    try:
        return os.stat(path).st_mtime_ns
    except FileNotFoundError:
        return 0


# -----------------------------
# Agent
# -----------------------------
class Agent(threading.Thread):
    def __init__(self, agent_id: int, iterations: int) -> None:
        super().__init__(name=f"agent-{agent_id}", daemon=False)
        self.agent_id = agent_id
        self.iterations = iterations

        self.modifications = 0
        self.touched_files: set[str] = set()
        self.blamed_counts: Counter[int] = Counter()
        self.active_runtime_s = 0.0

    def run(self) -> None:
        start = time.perf_counter()

        rng = random.Random()
        rng.seed((self.agent_id + 1) * 1_000_003 + int(start * 1e9))

        for _ in range(self.iterations):
            target_file = rng.choice(FILES)

            blamed_id = self.agent_id
            while blamed_id == self.agent_id:
                blamed_id = rng.randint(1, N_THREADS)

            lock = FILE_LOCKS[target_file]
            with lock:
                try:
                    with open(target_file, "r", encoding="utf-8") as f:
                        lines = f.readlines()
                except FileNotFoundError:
                    lines = []

                before_len = len(lines)
                lines = remove_agent_trace(lines, self.agent_id)
                removed = before_len - len(lines)
                lines = append_blame(lines, blamed_id)

                # --------- Hints for monitor (pre-write) ----------
                now_ns = time.time_ns()
                cur_thread = threading.current_thread()
                with LAST_WRITE_LOCK:
                    LAST_WRITE_HINTS[target_file] = {
                        "phase": "pre_write",
                        "when_ns": now_ns,
                        "thread_ident": threading.get_ident(),
                        "thread_id_hex": hex(id(cur_thread)),
                        "thread_name": cur_thread.name,
                        "agent_id": self.agent_id,
                    }

                # --------- Actual write ----------
                with open(target_file, "w", encoding="utf-8") as f:
                    f.writelines(lines)

                # --------- Hints for monitor (post-write) ----------
                now_ns2 = time.time_ns()
                with LAST_WRITE_LOCK:
                    LAST_WRITE_HINTS[target_file] = {
                        "phase": "post_write",
                        "when_ns": now_ns2,
                        "thread_ident": threading.get_ident(),
                        "thread_id_hex": hex(id(cur_thread)),
                        "thread_name": cur_thread.name,
                        "agent_id": self.agent_id,
                        # Optional: we could include counts for the monitor, but it recomputes via diff
                    }

            self.modifications += 1
            self.touched_files.add(target_file)
            self.blamed_counts[blamed_id] += 1

            # ---- Logging for this modification step ----
            log_event(
                f"{iso_now()} "
                f"event=modify "
                f"agent={self.agent_id} "
                f"file={target_file} "
                f"blamed={blamed_id} "
                f"removed_self_traces={removed} "
                f"total_mods_agent={self.modifications}"
            )

            time.sleep(rng.uniform(SLEEP_MIN, SLEEP_MAX))

        end = time.perf_counter()
        self.active_runtime_s = end - start

        self.print_summary()

    def print_summary(self) -> None:
        blamed_sorted = sorted(self.blamed_counts.items(), key=lambda x: (-x[1], x[0]))
        blamed_str = ", ".join([f"{tid}:{cnt}" for tid, cnt in blamed_sorted]) or "(none)"
        files_sorted = ", ".join(sorted(self.touched_files)) or "(none)"

        summary = (
            f"[Agent {self.agent_id}] "
            f"mods={self.modifications}, "
            f"files_touched={len(self.touched_files)} [{files_sorted}], "
            f"blamed={blamed_str}, "
            f"runtime_s={self.active_runtime_s:.6f}"
        )
        print(summary)

        # ---- Log summary too ----
        log_event(
            f"{iso_now()} "
            f"event=summary "
            f"agent={self.agent_id} "
            f"mods={self.modifications} "
            f"files_touched={len(self.touched_files)} "
            f"files=[{files_sorted}] "
            f"blamed_counts=[{blamed_str}] "
            f"runtime_s={self.active_runtime_s:.6f}"
        )


# -----------------------------
# Monitor (daemon thread)
# -----------------------------
class FileMonitor(threading.Thread):
    """
    A daemon thread that monitors the N_FILES text files. On detecting a change (via mtime),
    it diffs content against its last snapshot and logs a best-effort 'who did it' guess
    based on LAST_WRITE_HINTS timestamps and file names.
    """

    def __init__(self, files: List[str], agents: List[threading.Thread], poll_interval_s: float = MONITOR_POLL_INTERVAL_S) -> None:
        super().__init__(name="file-monitor", daemon=True)
        self.files = list(files)
        self.agents = agents
        self.poll_interval_s = poll_interval_s
        self._stop_event = threading.Event()
        # Cache: file -> (mtime_ns, [lines])
        self._cache: Dict[str, Tuple[int, List[str]]] = {}

    def init_cache(self) -> None:
        for path in self.files:
            self._cache[path] = (file_stat_mtime_ns(path), file_read_lines(path))

    def request_stop(self) -> None:
        self._stop_event.set()

    def anyone_alive(self) -> bool:
        return any(a.is_alive() for a in self.agents)

    def run(self) -> None:
        # Initialize cache at start
        self.init_cache()
        monitor_log(f"{iso_now()} event=monitor_start files={len(self.files)} poll_interval_s={self.poll_interval_s}")

        while not self._stop_event.is_set() and self.anyone_alive():
            self.scan_once()
            time.sleep(self.poll_interval_s)

        # One last sweep before terminating
        self.scan_once()
        monitor_log(f"{iso_now()} event=monitor_end")

    def scan_once(self) -> None:
        for path in self.files:
            try:
                mtime_ns = os.stat(path).st_mtime_ns
            except FileNotFoundError:
                mtime_ns = 0

            prev_mtime_ns, prev_lines = self._cache.get(path, (0, []))

            if mtime_ns > prev_mtime_ns:
                # File changed; read new content and compute a diff summary
                new_lines = file_read_lines(path)
                diff_lines = list(unified_diff(prev_lines, new_lines, fromfile=f"{path}:old", tofile=f"{path}:new", lineterm=""))
                diff_summary, added, removed = self._summarise_diff(diff_lines)

                # Try to guess the writer based on LAST_WRITE_HINTS within a time window
                guess = self._guess_writer(path, mtime_ns)

                # Log the event
                monitor_log(
                    f"{iso_now()} event=fs_modify file={path} "
                    f"mtime_ns={mtime_ns} "
                    f"guess_thread_name={guess.get('thread_name','?')} "
                    f"guess_thread_ident={guess.get('thread_ident','?')} "
                    f"guess_thread_memref={guess.get('thread_id_hex','?')} "
                    f"guess_agent_id={guess.get('agent_id','?')} "
                    f"added={added} removed={removed} "
                    f"diff_summary={diff_summary}"
                )

                # Update cache
                self._cache[path] = (mtime_ns, new_lines)

            elif path not in self._cache:
                # First time caching this file
                self._cache[path] = (mtime_ns, file_read_lines(path))

    def _guess_writer(self, path: str, change_mtime_ns: int) -> Dict[str, object]:
        with LAST_WRITE_LOCK:
            hint = LAST_WRITE_HINTS.get(path, None)
            if not hint:
                return {}

            when_ns = int(hint.get("when_ns", 0))
            # If the recorded hint is within window of observed change, treat as likely match
            if abs(change_mtime_ns - when_ns) <= MONITOR_TIME_WINDOW_NS:
                return {
                    "thread_ident": hint.get("thread_ident"),
                    "thread_id_hex": hint.get("thread_id_hex"),
                    "thread_name": hint.get("thread_name"),
                    "agent_id": hint.get("agent_id"),
                    "phase": hint.get("phase"),
                }
            else:
                return {}

    def _summarise_diff(self, diff_lines: List[str]) -> Tuple[str, int, int]:
        """
        Produce a compact summary: counts of '+'/'-' lines and a truncated diff preview.
        """
        added = sum(1 for ln in diff_lines if ln.startswith("+") and not ln.startswith("+++"))
        removed = sum(1 for ln in diff_lines if ln.startswith("-") and not ln.startswith("---"))

        # Truncate the diff for the log
        if len(diff_lines) > MONITOR_DIFF_MAX_LINES:
            preview = diff_lines[:MONITOR_DIFF_MAX_LINES] + ["... (diff truncated)"]
        else:
            preview = diff_lines

        # Escape newlines to keep monitor.txt line-per-event
        preview_joined = "\\n".join(preview)
        return preview_joined, added, removed


# -----------------------------
# Main
# -----------------------------
def main() -> None:
    ensure_files_exist()
    init_logs()

    log_event(f"{iso_now()} event=start n_threads={N_THREADS} n_files={N_FILES} iterations={ITERATIONS}")

    # Create agents
    agents = [Agent(agent_id=i, iterations=ITERATIONS) for i in range(1, N_THREADS + 1)]

    # Start monitor (daemon) before agents so it can catch the first changes
    monitor = FileMonitor(files=FILES, agents=agents, poll_interval_s=MONITOR_POLL_INTERVAL_S)
    monitor.start()

    # Start agents
    for a in agents:
        a.start()
    for a in agents:
        a.join()

    # Request monitor to stop, allow a last sweep, then join briefly
    monitor.request_stop()
    monitor.join(timeout=1.0)

    log_event(f"{iso_now()} event=end")


if __name__ == "__main__":
    main()

## Generate the dataset.json - part 2
Run the code below to generate the unified json file.

In [ ]:
# Jupyter cell: Incrementally unify monitor.txt and logs.txt into dataset.json,
# showing interactive progress and appending new events in batches.
#
# - Reads both files incrementally using byte offsets (no full reload).
# - Uses logs.txt total record count as a reference for progress.
# - After each batch, matches new monitor events to nearest runtime 'modify' (<= 250ms),
#   recomputes aggregates & ES search space, and rewrites dataset.json snapshot.
# - Persists state (file offsets, counts) to resume on re-run.

import os
import re
import json
import time
from datetime import datetime, timezone
from typing import Dict, List, Any, Optional, Tuple

# -----------------------------
# Paths & Settings
# -----------------------------
MONITOR_PATH = "monitor.txt"
RUNTIME_PATH = "logs.txt"
OUT_PATH     = "dataset.json"
STATE_PATH   = "dataset.state.json"   # persists offsets & counters across runs

BATCH_SLEEP_S   = 0.05   # small sleep between batches to keep UI responsive
BATCH_MAX_LINES = 2000   # max new lines per batch per file
MATCH_WINDOW_MS = 250    # max dt for monitor->runtime matching (can be tuned)

# -----------------------------
# Parsing utilities
# -----------------------------
ISO_TS_RE = re.compile(r"^(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{3}Z)\s+event=([^\s]+)\s*(.*)$")
KV_PAIR_RE = re.compile(r"(?P<key>[a-zA-Z0-9_\-]+)=(?P<val>[^ \t].*?)\b(?=\s+[a-zA-Z0-9_\-]+=|$)")
LIST_BRACKETS_RE = re.compile(r"^\[(.*)\]$")


def parse_iso8601_z(ts: str) -> datetime:
    return datetime.strptime(ts, "%Y-%m-%dT%H:%M:%S.%fZ").replace(tzinfo=timezone.utc)


def coerce_value(val: str) -> Any:
    v = val.strip()
    if (v.startswith('"') and v.endswith('"')) or (v.startswith("'") and v.endswith("'")):
        v = v[1:-1]
    m = LIST_BRACKETS_RE.match(v)
    if m:
        inner = m.group(1).strip()
        if not inner:
            return []
        parts = [p.strip() for p in inner.split(",")]
        return [coerce_value(p) for p in parts]
    if v.lower().startswith("0x"):
        return v
    try:
        return int(v)
    except ValueError:
        pass
    try:
        return float(v)
    except ValueError:
        pass
    if v.lower() in ("true", "false"):
        return v.lower() == "true"
    return v


def parse_kv_blob(blob: str) -> Dict[str, Any]:
    kvs: Dict[str, Any] = {}
    for m in KV_PAIR_RE.finditer(blob):
        kvs[m.group("key")] = coerce_value(m.group("val"))
    return kvs


def parse_event_line(line: str, source: str) -> Optional[Dict[str, Any]]:
    line = line.strip()
    if not line or line.startswith("#"):
        return None
    m = ISO_TS_RE.match(line)
    if not m:
        return None
    ts_iso, event_type, rest = m.groups()
    payload = parse_kv_blob(rest)
    return {
        "ts_iso": ts_iso,
        "ts_unix_ms": int(parse_iso8601_z(ts_iso).timestamp() * 1000),
        "source": source,
        "event": event_type,
        **payload,
    }


def count_reference_records(path: str) -> int:
    """Count *all* non-empty, non-comment lines in logs.txt as a progress baseline."""
    if not os.path.exists(path):
        return 0
    n = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            n += 1
    return n


# -----------------------------
# Incremental file reading (byte offsets)
# -----------------------------
def read_new_lines(path: str, start_offset: int, max_lines: int) -> Tuple[List[str], int]:
    """
    Read up to max_lines new lines from 'path' starting at byte offset 'start_offset'.
    Returns (lines, new_offset).
    """
    if not os.path.exists(path):
        return [], start_offset
    lines = []
    with open(path, "r", encoding="utf-8") as f:
        f.seek(start_offset)
        for _ in range(max_lines):
            line = f.readline()
            if not line:
                break
            lines.append(line)
        new_off = f.tell()
    return lines, new_off


# -----------------------------
# Matching & aggregation
# -----------------------------
def nearest_match(fs_evt: Dict[str, Any], runtime_events: List[Dict[str, Any]], max_dt_ms: int = 250) -> Optional[int]:
    fpath = fs_evt.get("file")
    t_ms = fs_evt.get("ts_unix_ms")
    best_idx, best_abs = None, None
    for i, e in enumerate(runtime_events):
        if e.get("event") != "modify":
            continue
        if e.get("file") != fpath:
            continue
        dt = abs(e.get("ts_unix_ms", 0) - t_ms)
        if best_abs is None or dt < best_abs:
            best_abs, best_idx = dt, i
    return best_idx if (best_abs is not None and best_abs <= max_dt_ms) else None


def aggregate(events_runtime: List[Dict[str, Any]], events_monitor: List[Dict[str, Any]]) -> Dict[str, Any]:
    per_agent: Dict[str, Dict[str, Any]] = {}
    per_file: Dict[str, Dict[str, Any]] = {}

    for e in events_runtime:
        if e.get("event") == "modify":
            agent = str(e.get("agent"))
            f = e.get("file")
            per_agent.setdefault(agent, {"mods": 0, "blame": {}, "files": set()})
            per_agent[agent]["mods"] += 1
            per_agent[agent]["files"].add(f)
            blamed = e.get("blamed")
            if blamed is not None:
                b = str(blamed)
                per_agent[agent]["blame"][b] = per_agent[agent]["blame"].get(b, 0) + 1

            per_file.setdefault(f, {"mods": 0, "agents": {}})
            per_file[f]["mods"] += 1
            per_file[f]["agents"][agent] = per_file[f]["agents"].get(agent, 0) + 1

    for a in per_agent.values():
        a["files"] = sorted(a["files"])

    monitor_guess_by_file: Dict[str, Dict[str, int]] = {}
    guess_window_diffs_ms: List[int] = []

    runtime_mods = [e for e in events_runtime if e.get("event") == "modify"]
    for m in events_monitor:
        if m.get("event") == "fs_modify":
            f = m.get("file")
            g_agent = m.get("guess_agent_id")
            monitor_guess_by_file.setdefault(f, {})
            if g_agent not in (None, "?", ""):
                g = str(g_agent)
                monitor_guess_by_file[f][g] = monitor_guess_by_file[f].get(g, 0) + 1
            idx = nearest_match(m, runtime_mods, max_dt_ms=10_000)
            if idx is not None:
                dt = abs(runtime_mods[idx]["ts_unix_ms"] - m["ts_unix_ms"])
                guess_window_diffs_ms.append(dt)

    return {
        "per_agent": per_agent,
        "per_file": per_file,
        "monitor_guess_by_file": monitor_guess_by_file,
        "match_dt_samples_ms": guess_window_diffs_ms,
    }


def suggest_es_space(aggr: Dict[str, Any]) -> Dict[str, Any]:
    space = {
        "poll_interval_s": {"type": "real", "min": 0.001, "max": 0.1, "log": True, "init": 0.01},
        "backoff_factor": {"type": "real", "min": 0.5, "max": 3.0},
        "matching_window_ms": {"type": "real", "min": 5.0, "max": 300.0},
        "file_priority_weights": {"type": "simplex", "labels": [], "init": []},
        "parse_thresholds": {
            "type": "dict",
            "removed_trace_weight": {"type": "real", "min": 0.0, "max": 2.0},
            "fs_added_weight": {"type": "real", "min": 0.0, "max": 2.0},
            "fs_removed_weight": {"type": "real", "min": 0.0, "max": 2.0},
        },
    }
    deltas = sorted(aggr.get("match_dt_samples_ms", []))
    space["matching_window_ms"]["init"] = float(
        max(5.0, min(300.0, (deltas[len(deltas)//2] * 1.5) if deltas else 50.0))
    )
    per_file = aggr.get("per_file", {})
    files = sorted(per_file.keys())
    mods = [per_file[f]["mods"] for f in files] if files else []
    total = float(sum(mods)) if mods else 0.0
    weights = [m / total for m in mods] if total > 0 else ([1.0 / len(files)] * len(files) if files else [])
    space["file_priority_weights"]["labels"] = files
    space["file_priority_weights"]["init"] = weights
    return space


# -----------------------------
# State, snapshot writing, and incremental engine
# -----------------------------
def load_state(path: str) -> Dict[str, Any]:
    if not os.path.exists(path):
        return {
            "monitor_offset": 0,
            "runtime_offset": 0,
            "events_monitor_count": 0,
            "events_runtime_count": 0
        }
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_state(path: str, state: Dict[str, Any]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)


def write_snapshot(out_path: str,
                   meta: Dict[str, Any],
                   events_monitor: List[Dict[str, Any]],
                   events_runtime: List[Dict[str, Any]],
                   aggregates: Dict[str, Any],
                   es_space: Dict[str, Any]) -> None:
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({
            "meta": meta,
            "events": {
                "monitor": events_monitor,
                "runtime": events_runtime
            },
            "aggregates": aggregates,
            "es_search_space": es_space,
        }, f, ensure_ascii=False, indent=2)


def process_incrementally(monitor_path: str,
                          runtime_path: str,
                          out_path: str,
                          state_path: str,
                          batch_max_lines: int = 2000,
                          match_window_ms: int = 250,
                          sleep_s: float = 0.05) -> Dict[str, Any]:
    # Preconditions
    if not os.path.exists(monitor_path):
        raise FileNotFoundError(f"Monitor log not found: {monitor_path}")
    if not os.path.exists(runtime_path):
        raise FileNotFoundError(f"Runtime log not found: {runtime_path}")

    # Load or init state
    state = load_state(state_path)
    mon_off = int(state.get("monitor_offset", 0))
    run_off = int(state.get("runtime_offset", 0))

    # Load existing snapshot events if any (to keep appending)
    events_monitor: List[Dict[str, Any]] = []
    events_runtime: List[Dict[str, Any]] = []
    if os.path.exists(out_path):
        try:
            with open(out_path, "r", encoding="utf-8") as f:
                prev = json.load(f)
            events_monitor = prev.get("events", {}).get("monitor", [])
            events_runtime = prev.get("events", {}).get("runtime", [])
        except Exception:
            # If previous dataset.json is corrupt, start fresh (but keep offsets)
            events_monitor = []
            events_runtime = []

    # Reference: total runtime lines for better user feedback
    ref_total_lines = count_reference_records(runtime_path)
    print(f"[INIT] logs.txt reference (non-empty, non-comment lines): {ref_total_lines}")

    total_added_monitor = 0
    total_added_runtime = 0

    # Run iterative batches until no new lines are found
    batch_idx = 0
    while True:
        batch_idx += 1
        # Read new lines up to BATCH_MAX_LINES for each file
        mon_lines, mon_off_new = read_new_lines(monitor_path, mon_off, batch_max_lines)
        run_lines, run_off_new = read_new_lines(runtime_path, run_off, batch_max_lines)

        if not mon_lines and not run_lines:
            print("[DONE] No new lines detected in either file. Stopping.")
            break

        # Parse lines into events
        new_monitor_events = []
        for ln in mon_lines:
            evt = parse_event_line(ln, source="monitor")
            if evt:
                new_monitor_events.append(evt)

        new_runtime_events = []
        for ln in run_lines:
            evt = parse_event_line(ln, source="runtime")
            if evt:
                new_runtime_events.append(evt)

        # Append & sort by time (keep arrays increasing by ts)
        if new_monitor_events:
            events_monitor.extend(new_monitor_events)
            events_monitor.sort(key=lambda e: e.get("ts_unix_ms", 0))

        if new_runtime_events:
            events_runtime.extend(new_runtime_events)
            events_runtime.sort(key=lambda e: e.get("ts_unix_ms", 0))

        # Matching only needs runtime 'modify' events, precompute list
        runtime_mods = [e for e in events_runtime if e.get("event") == "modify"]

        # Match newly added monitor events to nearest runtime 'modify'
        # (We only need to process the tail; but for correctness, re-check any monitor events that
        # do not yet have matched indexes.)
        matched_this_batch = 0
        for m in (events_monitor[-len(new_monitor_events):] if new_monitor_events else []):
            if m.get("event") != "fs_modify":
                continue
            idx = nearest_match(m, runtime_mods, max_dt_ms=match_window_ms)
            if idx is not None:
                m["matched_runtime_idx"] = idx
                m["matched_runtime_ts_ms"] = runtime_mods[idx]["ts_unix_ms"]
                m["matched_dt_ms"] = abs(runtime_mods[idx]["ts_unix_ms"] - m["ts_unix_ms"])
                matched_this_batch += 1

        # Aggregation & ES space suggestion
        aggregates = aggregate(events_runtime, events_monitor)
        es_space = suggest_es_space(aggregates)

        # Build/Update meta
        meta = {
            "monitor_path": monitor_path,
            "runtime_path": runtime_path,
            "generated_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ"),
            "notes": "Unified dataset for Evolutionary Strategies over monitoring parameters (incremental).",
            "state": {
                "monitor_offset": mon_off_new,
                "runtime_offset": run_off_new,
                "batches_completed": batch_idx
            }
        }

        # Rewrite snapshot (append semantics by snapshotting full arrays)
        write_snapshot(out_path, meta, events_monitor, events_runtime, aggregates, es_space)

        # Update state (offsets) and persist
        mon_off, run_off = mon_off_new, run_off_new
        state["monitor_offset"] = mon_off
        state["runtime_offset"] = run_off
        state["events_monitor_count"] = len(events_monitor)
        state["events_runtime_count"] = len(events_runtime)
        save_state(state_path, state)

        # Stats & user feedback
        total_added_monitor += len(new_monitor_events)
        total_added_runtime += len(new_runtime_events)

        # Progress against reference (runtime)
        processed_runtime_lines = count_reference_records(runtime_path)  # recount to include concurrently appended lines
        ref = processed_runtime_lines if processed_runtime_lines > 0 else ref_total_lines
        pct = (len(events_runtime) / ref * 100.0) if ref > 0 else 0.0

        print(
            f"[BATCH {batch_idx:03d}] "
            f"+monitor:{len(new_monitor_events)} (+{matched_this_batch} matched) "
            f"+runtime:{len(new_runtime_events)} | "
            f"Totals -> monitor:{len(events_monitor)}, runtime:{len(events_runtime)} "
            f"| runtime~progress:{pct:.1f}% of logs.txt (events vs lines)\n"
            f"          Snapshot: {out_path}  State: {state_path}"
        )

        # Small sleep to keep UI responsive & allow new lines to accumulate
        time.sleep(sleep_s)

    return {
        "events_counts": {"monitor": len(events_monitor), "runtime": len(events_runtime)},
        "out_path": out_path,
        "state_path": state_path
    }


# -----------------------------
# Run
# -----------------------------
summary = process_incrementally(
    monitor_path=MONITOR_PATH,
    runtime_path=RUNTIME_PATH,
    out_path=OUT_PATH,
    state_path=STATE_PATH,
    batch_max_lines=BATCH_MAX_LINES,
    match_window_ms=MATCH_WINDOW_MS,
    sleep_s=BATCH_SLEEP_S
)

print(
    "\n[SUMMARY] Unified dataset:",
    summary["out_path"],
    "| State:", summary["state_path"],
    "| Counts -> monitor:", summary["events_counts"]["monitor"], "runtime:", summary["events_counts"]["runtime"]
)

# Advanced task - Evolutionary Optimisation of Monitoring Strategies
### (Using the unified `dataset.json` as the Data Source)

Your multithreaded actor system, together with the file‑system monitoring thread, produces a **structured dataset** stored in **`dataset.json`**. This dataset is generated by merging:

- The runtime modification log (`logs.txt`)
- The monitor's file‑system modification log (`monitor.txt`)
- Automated alignment between monitor events and runtime events
- Aggregated statistics
- A pre‑computed evolutionary search space

Because the system exhibits:

- **Random timing jitter**  
- **Concurrent nondeterministic writes to shared files**  
- **High‑frequency modifications**  
- **Noise in both the runtime log and the monitor observations**  

…a naïve monitor is unable to reconstruct per‑agent behaviour reliably.

Your task is to design a **monitoring strategy** whose parameters are optimised through an **Evolutionary Algorithm (EA)** such as:

- Genetic Algorithms (GA)
- Evolution Strategies (ES)
- CMA‑ES
- Hybrid schemes

This monitoring strategy must operate **exclusively on the structured contents of `dataset.json`**.

## The Unified Dataset: `dataset.json`

The dataset includes four key sections:

### **1. `events.runtime`**
Derived from the actor threads:
- `event=modify`
- `agent`
- `file`
- `blamed`
- `removed_self_traces`
- `total_mods_agent`
- Timestamps in ISO8601 and Unix ms

### **2. `events.monitor`**
Derived from the monitor thread:
- `event=fs_modify`
- Modified `file`
- Diff-based info:
  - `added`
  - `removed`
  - `diff_summary`
- Monitor’s guess of responsible agent:
  - `guess_agent_id`
  - `guess_thread_ident`
  - `guess_thread_memref`
- Timestamp alignment info:
  - `matched_runtime_idx`
  - `matched_runtime_ts_ms`
  - `matched_dt_ms`

### **3. `aggregates`**
Ground‑truth summary statistics:
- `per_agent`:
  - true modification counts
  - files touched
  - blame distribution
- `per_file`:
  - modification hotspots
  - agent contributions
- Monitor guess behaviour:
  - `monitor_guess_by_file`
- Timestamp matching errors:
  - `match_dt_samples_ms`

### **4. `es_search_space`**
A suggested initial evolutionary search space containing:
- Bounds
- Default values
- Parameter types (real, simplex, dict, etc.)

## Evolutionary Approaches

You may follow either or both:

### **Genetic Algorithms (GA)**
Optimise discrete components such as:
- Ordering of parsing heuristics
- Boolean flags for enabling/disabling certain monitors
- File‑target prioritisation
- Category‑based parsing decisions

### **Evolution Strategies (ES)**
Optimise continuous parameters:
- `poll_interval_s`
- `matching_window_ms`
- Backoff timing
- Thresholds for interpreting diffs
- Noise‑handling weights
- Simplex‑based per‑file priority weights

## Evolvable Parameters (using `dataset.json` fields)

Your EA may evolve any subset of:

### **Sampling / Polling Behaviour**
- `poll_interval_s`: how often a virtual monitor would scan the logs  
- `backoff_factor`: adjust polling based on activity

### **Event Selection Weights**
- Weight for `fs_modify` events  
- Weight for `runtime.modify` events  
- Weight for high‑diff events (many added/removed lines)

### **Matching Parameters**
Using `matched_dt_ms` and `match_dt_samples_ms`:
- Dynamic window for matching `fs_modify` → `modify`
- Confidence derived from timestamp proximity

### **Noise Interpretation Heuristics**
- How much to trust monitor guesses (`guess_agent_id`)
- Thresholds for ignoring mismatches
- Diff‑based inference weights:
  - `removed_trace_weight`
  - `fs_added_weight`
  - `fs_removed_weight`

### **File Priority Weights**
Simplex vector over `file_priority_weights.labels` with initial values in `file_priority_weights.init`.

## Fitness Function  
Your fitness must be computed *entirely* using information inside `dataset.json`.

### You should evaluate how well your reconstructed behaviour matches the ground truth in `aggregates`:

### **1. Per‑Agent Modification Accuracy**
Compare:
- Your inferred modifications per agent  
- Against `aggregates.per_agent[agent]["mods"]`

### **2. Blame Pattern Accuracy**
Compare:
- Your inferred blame matrix  
- Against `aggregates.per_agent[agent]["blame"]`

### **3. File Hotspot Accuracy**
Compare:
- Reconstructed file activity  
- Against `aggregates.per_file[file]["mods"]`

### **4. Alignment Error Quality**
Evaluate:
- How well your evolved matching rules reduce error in `matched_dt_ms`

### **5. (Optional) Efficiency Penalty**
Reward:
- Lower polling frequency  
- Fewer unnecessary computations  

Your fitness may be:
- Weighted sum  
- Multi‑objective (accuracy vs cost)  
- Pareto‑front optimisation (NSGA‑II)

# Part 4: Analysis & Evaluation

Your notebook must include:


## 1. Description of the Multithreaded System

### Architecture
Use `events.runtime` and `events.monitor` to describe:
- Agents  
- Files  
- Monitor thread  
- Logged events

### Concurrency Challenges
Using `match_dt_samples_ms` illustrate:
- Race conditions  
- Interleaving writes  
- Noise from filesystem delays

### File Interaction Behaviour
Using `aggregates.per_file`:
- Identify hotspots  
- Examine distribution of modifications

## 2. Description of Your Monitoring Strategy

### Observable Inputs
From `dataset.json`:
- Runtime events  
- Monitor events  
- Pre‑alignment data  
- Aggregates (for ground truth)

### Hidden Behaviour
What you *cannot* observe:
- True thread scheduling  
- Exact write timing  
- Internal state transitions

### Reconstruction Strategy
Describe:
- How evolved parameters guide inference  
- How you merge monitor and runtime signals  
- How you weight conflicting evidence  
- Any smoothing or denoising mechanisms

## 3. Evolutionary Optimisation Methodology

### Genotype Representation
Document which fields from `es_search_space` you evolve:
- Continuous
- Discrete
- Simplex vectors
- Hybrid encodings

### Variation Operators
- Mutation  
- Crossover  
- Adaptive mutation  
- Restart policies  

### Fitness Function
Detail formulas for:
- Modification accuracy  
- Blame accuracy  
- Hotspot score  
- Efficiency penalties  

### Selection Scheme
- Tournament  
- Rank selection  
- Elitism  
- Multi‑objective (optional)

## 4. Quantitative Results

You must include:

### Convergence Plots
- Fitness over generations  
- Parameter trajectories  

### Comparisons
- Naïve strategy vs EA‑optimised  
- Ablations (remove parameters, test impact)

### Accuracy Results
Compare your reconstructed output to:
- `aggregates.per_agent`
- `aggregates.per_file`
- `aggregates.monitor_guess_by_file`
- Time‑matching accuracy

## 5. Critical Reflection

Discuss:
- Which behaviours were easy/hard to infer  
- Which parameters mattered most  
- Role of noise and timestamp uncertainty  
- Surprising EA behaviours  
- Whether another EA method might perform better  
- Applicability to real-world log monitoring  
